In [1]:
import os
import sys
import json
from datetime import datetime

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = (
    "/scratch/shareddata/dldata/huggingface-hub-cache/hub"  # Load local models
)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = (
    f"{os.environ["WRKDIR"]}/.cache/huggingface"  # Cache in work directory
)

from vllm import LLM, SamplingParams
import transformers
import numpy as np
from datasets import load_dataset

from utils.helpers import (
    get_system_prompt,
    get_default_response,
    make_prompt,
    parse_output,
    sample_dataset
)

# Force reload changes
from utils.prompts import *

from utils.constants import (
    PIPE_MAX_NEW_TOKENS,
    MODEL_TEMPERATURE,
    BATCH_SIZE,
    PIPE_RETURN_FULL_TEXT,
    DEFAULT_MODEL
)

INFO 04-02 22:05:42 [__init__.py:216] Automatically detected platform cuda.


In [2]:
LABELS = ["yes", "no"]  # Labels per column
EVAL_COLS = [
    "The exercise description matched the selected theme (Yes/No)",
    "The exercise description matched the selected topic (Yes/No)",
    "Included concepts that were too advanced (Yes/No)",
]
PRED_COLS = ["augmentedProblemDescription", "augmentedExampleSolution"]

DEFAULT_DATA = "./data/final_dataset.csv"

In [3]:
task = "detect"
n_rows = -1
use_instructions = True
seed = 42
number_of_demonstrations = 0
type_of_demonstrations = 0

dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")
dataset = dataset.shuffle(seed=seed)

demonstrations = sample_dataset(
    dataset, seed, 0, type_of_demonstrations
)

system_prompt = get_system_prompt(task, demonstrations, bool(use_instructions))

dataset = dataset.map(
    lambda row: {
        "user_prompt": make_prompt(row, task),
        "system_prompt": system_prompt,
    },
)

if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

In [4]:
model = 'mistralai/Mistral-7B-Instruct-v0.3'
mode = "auto"
llm = LLM(model=model,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    max_num_seqs=8,
    enforce_eager=True,
    tokenizer_mode=mode,
    config_format=mode,
    load_format=mode
)

INFO 04-02 22:05:47 [arg_utils.py:504] HF_HUB_OFFLINE is True, replace model_id [mistralai/Mistral-7B-Instruct-v0.3] to model_path [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/0d4b76e1efeb5eb6f6b5e757c79870472e04bd3a]
INFO 04-02 22:05:47 [arg_utils.py:504] HF_HUB_OFFLINE is True, replace model_id [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/0d4b76e1efeb5eb6f6b5e757c79870472e04bd3a] to model_path [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/0d4b76e1efeb5eb6f6b5e757c79870472e04bd3a]
INFO 04-02 22:05:47 [utils.py:233] non-default args: {'max_model_len': 4096, 'max_num_seqs': 8, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/0d4b76e1efeb5eb6f6b5e757c79870472e04bd3a'}
INFO 04-02 22:05:47

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 04-02 22:05:47 [model.py:1682] Your device 'Tesla V100-PCIE-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-02 22:05:47 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 04-02 22:05:47 [model.py:1510] Using max model len 4096
INFO 04-02 22:05:50 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-02 22:05:50 [__init__.py:381] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=2345643) INFO 04-02 22:05:50 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=2345643) INFO 04-02 22:05:50 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/0d4b76e1efeb5eb6f6b5e757c79870472e04bd3a', speculative_config=None, tokenizer='/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--mistralai--Mistral-7B-Ins

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:04 [default_loader.py:267] Loading weights took 10.53 seconds
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:05 [gpu_model_runner.py:2653] Model loading took 13.5084 GiB and 10.709638 seconds
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:07 [gpu_worker.py:298] Available KV cache memory: 14.17 GiB
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:08 [kv_cache_utils.py:1087] GPU KV cache size: 116,048 tokens
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:08 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 28.33x
(EngineCore_DP0 pid=2345643) WARNING 04-02 22:06:08 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:08 [core.py:210] init engine (profile, create kv cache, warmup model) took 2.87 seconds
(EngineCore_DP0 pid=2345643) INFO 04-02 22:06:08 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 04-02 22:06:08 [

(EngineCore_DP0 pid=2345643) CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


In [5]:
sampling_params = SamplingParams(temperature=0.3, max_tokens=64, top_p=0.8, top_k=20, min_p=0)

default_response = get_default_response(task)
results = {key: [] for key in default_response.keys()}

batch_size = BATCH_SIZE
user_prompts = dataset["user_prompt"]
system_prompts = dataset["system_prompt"]

prompts = [
        [
            {"role": "system", "content": sp},
            {"role": "user", "content": up},
        ]
        for sp, up in zip(system_prompts, user_prompts)
]

tokenizer = transformers.AutoTokenizer.from_pretrained(model)

prompts = [
    tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    ) for prompt in prompts]

output = llm.generate(prompts, sampling_params, use_tqdm=True)

Adding requests:   0%|          | 0/535 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/535 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
for out in output:
    text = out.outputs[0].text
    parsed = parse_output(text)
    for key, value in default_response.items():
        results[key].append(json.dumps(parsed.get(key, value)))

for column_name, column_data in results.items():
    dataset = dataset.add_column(column_name, column_data)

In [8]:
print(results)

{'themeCorrect': ['"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"no"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"no"', '"no"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"no"', '"no"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"no"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"no"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"', '"yes"',